In [66]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

In [67]:
cursor.executescript("""
-- 学生表
CREATE TABLE Student(
    SId varchar(10),
    Sname varchar(10),
    Sage datetime,
    Ssex varchar(10)
);

-- 课程表
CREATE TABLE Course(
    CId varchar(10),
    Cname nvarchar(10),
    TId varchar(10)
);

-- 教师表
CREATE TABLE Teacher(
    TId varchar(10),
    Tname varchar(10)
);

-- 成绩表
CREATE TABLE SC(
    SId varchar(10),
    CId varchar(10),
    score decimal(18,1)
);
""")

cursor.executescript("""
-- 学生数据
INSERT INTO Student VALUES
('01', '赵雷', '1990-01-01', '男'),
('02', '钱电', '1990-12-21', '男'),
('03', '孙风', '1990-12-20', '男'),
('04', '李云', '1990-12-06', '男'),
('05', '周梅', '1991-12-01', '女'),
('06', '吴兰', '1992-01-01', '女'),
('07', '郑竹', '1989-01-01', '女'),
('09', '张三', '2017-12-20', '女'),
('10', '李四', '2017-12-25', '女'),
('11', '李四', '2012-06-06', '女'),
('12', '赵六', '2013-06-13', '女'),
('13', '孙七', '2014-06-01', '女');

-- 课程数据
INSERT INTO Course VALUES
('01', '语文', '02'),
('02', '数学', '01'),
('03', '英语', '03');

-- 教师数据
INSERT INTO Teacher VALUES
('01', '张三'),
('02', '李四'),
('03', '王五');

-- 成绩数据
INSERT INTO SC VALUES
('01', '01', 80),
('01', '02', 90),
('01', '03', 99),
('02', '01', 70),
('02', '02', 60),
('02', '03', 80),
('03', '01', 80),
('03', '02', 80),
('03', '03', 80),
('04', '01', 50),
('04', '02', 30),
('04', '03', 20),
('05', '01', 76),
('05', '02', 87),
('06', '01', 31),
('06', '03', 34),
('07', '02', 89),
('07', '03', 98);
""")

In [68]:
def print_query_result(description, query):
    print(f"\n=== {description} ===")
    cursor.execute(query)
    

    col_names = [desc[0] for desc in cursor.description]
    print(" | ".join(col_names))
    
    for row in cursor.fetchall():
        print(" | ".join(map(str, row)))

1.查询" 01 "课程⽐" 02 "课程成绩⾼的学⽣的信息及课程分数

In [69]:
query_1 = """
SELECT 
    stu.*,
    s1.score AS "01_score",
    s2.score AS "02_score"
FROM 
    Student stu
JOIN 
    (SELECT sid, score FROM SC WHERE CId = '01') AS s1 ON stu.SId = s1.SId
JOIN 
    (SELECT sid, score FROM SC WHERE CId = '02') AS s2 ON stu.SId = s2.SId
WHERE 
    s1.score > s2.score;
"""
print_query_result("查询01课程⽐02课程成绩⾼的学⽣的信息及课程分数", query_1)


=== 查询01课程⽐02课程成绩⾼的学⽣的信息及课程分数 ===
SId | Sname | Sage | Ssex | 01_score | 02_score
02 | 钱电 | 1990-12-21 | 男 | 70 | 60
04 | 李云 | 1990-12-06 | 男 | 50 | 30


2.查询同时存在" 01 "课程和" 02 "课程的情况

In [70]:
query_2 = """
SELECT 
    s1.SId,
    s1.score AS "01_score",
    s2.score AS "02_score" 
FROM 
    (SELECT SId, score FROM SC WHERE CId = '01') s1
JOIN 
    (SELECT SId, score FROM SC WHERE CId = '02') s2
ON s1.SId = s2.SId;
"""
print_query_result("同时存在 01 课程和 02 课程的情况", query_2)


=== 同时存在 01 课程和 02 课程的情况 ===
SId | 01_score | 02_score
01 | 80 | 90
02 | 70 | 60
03 | 80 | 80
04 | 50 | 30
05 | 76 | 87


3.查询存在" 01 "课程但可能不存在" 02 "课程的情况

In [71]:
query_3 = """
SELECT 
    s1.sid,
    s1.score AS "01_score",
    s2.score AS "02_score" 
FROM 
    (SELECT sid, score FROM SC WHERE CId = '01') AS s1 
    LEFT JOIN 
    (SELECT sid, score FROM SC WHERE CId = '02') AS s2 
    ON s1.sid = s2.sid;
"""
print_query_result("查询存在01课程但可能不存在02课程的情况", query_3)


=== 查询存在01课程但可能不存在02课程的情况 ===
sid | 01_score | 02_score
01 | 80 | 90
02 | 70 | 60
03 | 80 | 80
04 | 50 | 30
05 | 76 | 87
06 | 31 | None


4.查询不存在" 01 "课程但存在" 02 "课程的情况

In [72]:
query_4 = """
SELECT 
    s2.sid,
    s1.score AS "01_score",
    s2.score AS "02_score" 
FROM 
    (SELECT sid, score FROM SC WHERE CId = '01') AS s1 
    RIGHT JOIN 
    (SELECT sid, score FROM SC WHERE CId = '02') AS s2 
    ON s1.sid = s2.sid;
"""
print_query_result("查询不存在01课程但存在02课程的情况", query_4)


=== 查询不存在01课程但存在02课程的情况 ===
sid | 01_score | 02_score
01 | 80 | 90
02 | 70 | 60
03 | 80 | 80
04 | 50 | 30
05 | 76 | 87
07 | None | 89


5.查询平均成绩⼤于等于 60 分的同学的学⽣编号和学⽣姓名和平均成绩

In [73]:
query_5 = """
SELECT 
    stu.sid,
    stu."sname",
    s1.avg_score 
FROM 
    Student AS stu 
    JOIN 
    (SELECT sid, AVG(score) AS avg_score FROM SC GROUP BY sid) AS s1 
    ON stu.sid = s1.sid 
WHERE 
    s1.avg_score > 60;
"""
print_query_result("查询平均成绩⼤于等于 60 分的同学的学⽣编号和学⽣姓名和平均成绩", query_5)


=== 查询平均成绩⼤于等于 60 分的同学的学⽣编号和学⽣姓名和平均成绩 ===
SId | Sname | avg_score
01 | 赵雷 | 89.66666666666667
02 | 钱电 | 70.0
03 | 孙风 | 80.0
05 | 周梅 | 81.5
07 | 郑竹 | 93.5


6.查询在 SC 表存在成绩的学⽣信息

In [74]:
query_6 = """
SELECT DISTINCT stu.* 
FROM 
    Student AS stu, 
    SC 
WHERE 
    stu.SId = SC.SId;
"""
print_query_result("查询在 SC 表存在成绩的学⽣信息", query_6)


=== 查询在 SC 表存在成绩的学⽣信息 ===
SId | Sname | Sage | Ssex
01 | 赵雷 | 1990-01-01 | 男
02 | 钱电 | 1990-12-21 | 男
03 | 孙风 | 1990-12-20 | 男
04 | 李云 | 1990-12-06 | 男
05 | 周梅 | 1991-12-01 | 女
06 | 吴兰 | 1992-01-01 | 女
07 | 郑竹 | 1989-01-01 | 女


7.查询所有同学的学⽣编号、学⽣姓名、选课总数、所有课程的总成绩

In [75]:
query_7 = """
SELECT 
    stu.SId, 
    stu.Sname, 
    COUNT(sc.SId) AS "选课总数", 
    SUM(sc.score) AS "所有课程的总成绩" 
FROM 
    Student AS stu 
    LEFT JOIN SC ON stu.SId = sc.SId 
GROUP BY 
    stu.SId, 
    stu.Sname;
"""
print_query_result("查询所有同学的学⽣编号、学⽣姓名、选课总数、所有课程的总成绩", query_7)


=== 查询所有同学的学⽣编号、学⽣姓名、选课总数、所有课程的总成绩 ===
SId | Sname | 选课总数 | 所有课程的总成绩
01 | 赵雷 | 3 | 269
02 | 钱电 | 3 | 210
03 | 孙风 | 3 | 240
04 | 李云 | 3 | 100
05 | 周梅 | 2 | 163
06 | 吴兰 | 2 | 65
07 | 郑竹 | 2 | 187
09 | 张三 | 0 | None
10 | 李四 | 0 | None
11 | 李四 | 0 | None
12 | 赵六 | 0 | None
13 | 孙七 | 0 | None


8.查询「李」姓老师的数量

In [76]:
query_8 = """
SELECT COUNT(*) 
FROM Teacher 
WHERE Tname LIKE '李%';
"""
print_query_result("查询「李」姓老师的数量", query_8)


=== 查询「李」姓老师的数量 ===
COUNT(*)
1


9.查询学过「张三」老师授课的同学的信息

In [77]:
query_9 = """
SELECT s1.* 
FROM 
    (SELECT stu.*, sc.CId FROM Student AS stu JOIN SC ON stu.SId = sc.SId) AS s1 
    JOIN 
    (SELECT Teacher.Tname, Course.CId FROM Course JOIN Teacher ON Course.TId = Teacher.TId) AS c1 
    ON s1.CId = c1.CId 
WHERE 
    c1.Tname = '张三';
"""
print_query_result("查询学过「张三」老师授课的同学的信息", query_9)


=== 查询学过「张三」老师授课的同学的信息 ===
SId | Sname | Sage | Ssex | CId
01 | 赵雷 | 1990-01-01 | 男 | 02
02 | 钱电 | 1990-12-21 | 男 | 02
03 | 孙风 | 1990-12-20 | 男 | 02
04 | 李云 | 1990-12-06 | 男 | 02
05 | 周梅 | 1991-12-01 | 女 | 02
07 | 郑竹 | 1989-01-01 | 女 | 02


10.查询没有学全所有课程的同学的信息

In [78]:
query_10 = """
SELECT stu.* 
FROM Student AS stu 
WHERE SId NOT IN 
    (SELECT s1.sid 
     FROM 
         (SELECT sid, COUNT(sid) AS count_sid 
          FROM SC 
          GROUP BY sid) AS s1 
     WHERE s1.count_sid = 3);
"""
print_query_result("查询没有学全所有课程的同学的信息", query_10)


=== 查询没有学全所有课程的同学的信息 ===
SId | Sname | Sage | Ssex
05 | 周梅 | 1991-12-01 | 女
06 | 吴兰 | 1992-01-01 | 女
07 | 郑竹 | 1989-01-01 | 女
09 | 张三 | 2017-12-20 | 女
10 | 李四 | 2017-12-25 | 女
11 | 李四 | 2012-06-06 | 女
12 | 赵六 | 2013-06-13 | 女
13 | 孙七 | 2014-06-01 | 女


11.查询至少有一门课与学号为" 01 "的同学所学相同的同学的信息

In [79]:
query_11 = """
SELECT DISTINCT stu.* 
FROM 
    Student AS stu 
    JOIN SC ON stu.SId = SC.SId 
WHERE 
    SC.CId IN 
        (SELECT CId 
         FROM SC 
         WHERE SId = '01');
"""
print_query_result("查询至少有一门课与学号为01的同学所学相同的同学的信息", query_11)


=== 查询至少有一门课与学号为01的同学所学相同的同学的信息 ===
SId | Sname | Sage | Ssex
01 | 赵雷 | 1990-01-01 | 男
02 | 钱电 | 1990-12-21 | 男
03 | 孙风 | 1990-12-20 | 男
04 | 李云 | 1990-12-06 | 男
05 | 周梅 | 1991-12-01 | 女
06 | 吴兰 | 1992-01-01 | 女
07 | 郑竹 | 1989-01-01 | 女


12.查询和" 01 "号的同学学习的课程 完全相同的其他同学的信息

In [80]:
query_12 = """
SELECT stu.* 
FROM 
    Student AS stu 
    JOIN 
    (SELECT s2.sid 
     FROM 
         SC AS s1 
         JOIN SC AS s2 ON s1.CId = s2.CId AND s1.SId = '01' AND s2.SId != '01' 
     GROUP BY s2.sid 
     HAVING COUNT(s2.CId) = (SELECT COUNT(*) FROM SC WHERE SId = '01')) AS s 
    ON stu.SId = s.sid;
"""
print_query_result("查询和01号的同学学习的课程 完全相同的其他同学的信息", query_12)


=== 查询和01号的同学学习的课程 完全相同的其他同学的信息 ===
SId | Sname | Sage | Ssex
02 | 钱电 | 1990-12-21 | 男
03 | 孙风 | 1990-12-20 | 男
04 | 李云 | 1990-12-06 | 男


13.查询没学过"张三"老师讲授的任⼀门课程的学生姓名

In [81]:
query_13 = """
SELECT * 
FROM Student 
WHERE SId NOT IN 
    (SELECT s1.sid 
     FROM 
         (SELECT stu.*, sc.CId FROM Student AS stu JOIN SC ON stu.SId = sc.SId) AS s1 
         JOIN 
         (SELECT Teacher.Tname, Course.CId FROM Course JOIN Teacher ON Course.TId = Teacher.TId) AS c1 
         ON s1.CId = c1.CId 
     WHERE c1.Tname = '张三');
"""
print_query_result("查询没学过张三老师讲授的任⼀门课程的学生姓名", query_13)


=== 查询没学过张三老师讲授的任⼀门课程的学生姓名 ===
SId | Sname | Sage | Ssex
06 | 吴兰 | 1992-01-01 | 女
09 | 张三 | 2017-12-20 | 女
10 | 李四 | 2017-12-25 | 女
11 | 李四 | 2012-06-06 | 女
12 | 赵六 | 2013-06-13 | 女
13 | 孙七 | 2014-06-01 | 女


14.查询两门及其以上不及格课程的同学的学号，姓名及其平均成绩

In [82]:
query_14 = """
SELECT 
    stu.Sname, 
    stu.SId, 
    s1.avg_score 
FROM 
    Student AS stu 
    JOIN 
    (SELECT sid, AVG(score) AS avg_score 
     FROM SC 
     WHERE score < 60 
     GROUP BY sid 
     HAVING COUNT(*) >= 2) AS s1 
    ON stu.SId = s1.SId;
    """
print_query_result("查询两门及其以上不及格课程的同学的学号，姓名及其平均成绩", query_14)


=== 查询两门及其以上不及格课程的同学的学号，姓名及其平均成绩 ===
Sname | SId | avg_score
李云 | 04 | 33.333333333333336
吴兰 | 06 | 32.5


15.检索" 01 "课程分数⼩于 60，按分数降序排列的学生信息16.按平均成绩从⾼到低显示所有学生的所有课程的成绩以及平均成绩

In [83]:
query_15 = """
SELECT * 
FROM Student 
WHERE SId IN 
    (SELECT SId 
     FROM SC 
     WHERE CId = '01' AND score < 60 
     ORDER BY score DESC);
     """
print_query_result("检索01课程分数⼩于 60，按分数降序排列的学生信息16.按平均成绩从⾼到低显示所有学生的所有课程的成绩以及平均成绩", query_15)


=== 检索01课程分数⼩于 60，按分数降序排列的学生信息16.按平均成绩从⾼到低显示所有学生的所有课程的成绩以及平均成绩 ===
SId | Sname | Sage | Ssex
04 | 李云 | 1990-12-06 | 男
06 | 吴兰 | 1992-01-01 | 女


16.按平均成绩从⾼到低显示所有学生的所有课程的成绩以及平均成绩

In [84]:
query_16 = """
SELECT sc.*, s2.avg_score 
FROM SC 
JOIN 
    (SELECT sid, AVG(score) AS avg_score 
     FROM SC 
     GROUP BY sid) AS s2 
ON sc.sid = s2.sid 
ORDER BY s2.avg_score DESC;
"""
print_query_result("按平均成绩从⾼到低显示所有学生的所有课程的成绩以及平均成绩", query_16)


=== 按平均成绩从⾼到低显示所有学生的所有课程的成绩以及平均成绩 ===
SId | CId | score | avg_score
07 | 02 | 89 | 93.5
07 | 03 | 98 | 93.5
01 | 01 | 80 | 89.66666666666667
01 | 02 | 90 | 89.66666666666667
01 | 03 | 99 | 89.66666666666667
05 | 01 | 76 | 81.5
05 | 02 | 87 | 81.5
03 | 01 | 80 | 80.0
03 | 02 | 80 | 80.0
03 | 03 | 80 | 80.0
02 | 01 | 70 | 70.0
02 | 02 | 60 | 70.0
02 | 03 | 80 | 70.0
04 | 01 | 50 | 33.333333333333336
04 | 02 | 30 | 33.333333333333336
04 | 03 | 20 | 33.333333333333336
06 | 01 | 31 | 32.5
06 | 03 | 34 | 32.5


17.查询各科成绩最高分、最低分和平均分： 以如下形式显示：课程 ID，课程 name，最高分，最低分，平均分，及格率，中等率，优良率，优秀率 及格为>=60，中等为：70-80，优良为：80-90，优秀为：>=90 要求输出课程号和选修⼈数，查询结果按⼈数降序排列，若⼈数相同，按课程号升序排列

In [85]:
query_17 = """
SELECT 
    sc.CId, 
    course.Cname, 
    MAX(sc.score) AS "最高分", 
    MIN(sc.score) AS "最低分", 
    AVG(sc.score) AS "平均分", 
    COUNT(sc.CId) AS "选修人数", 
    SUM(CASE WHEN sc.score >= 60 THEN 1 ELSE 0 END) * 1.0 / COUNT(sc.CId) AS "及格率", 
    SUM(CASE WHEN sc.score >= 70 AND sc.score < 80 THEN 1 ELSE 0 END) * 1.0 / COUNT(sc.CId) AS "中等率", 
    SUM(CASE WHEN sc.score >= 80 AND sc.score < 90 THEN 1 ELSE 0 END) * 1.0 / COUNT(sc.CId) AS "优良率", 
    SUM(CASE WHEN sc.score >= 90 THEN 1 ELSE 0 END) * 1.0 / COUNT(sc.CId) AS "优秀率" 
FROM 
    SC, Course 
WHERE 
    sc.CId = course.CId 
GROUP BY 
    sc.CId, course.Cname 
ORDER BY 
    "选修人数" DESC, sc.CId;
    """
print_query_result("查询各科成绩最高分、最低分和平均分： 以如下形式显示：课程 ID，课程 name，最高分，最低分，平均分，及格率，中等率，优良率，优秀率 及格为>=60，中等为：70-80，优良为：80-90，优秀为：>=90 要求输出课程号和选修⼈数，查询结果按⼈数降序排列，若⼈数相同，按课程号升序排列", query_17)


=== 查询各科成绩最高分、最低分和平均分： 以如下形式显示：课程 ID，课程 name，最高分，最低分，平均分，及格率，中等率，优良率，优秀率 及格为>=60，中等为：70-80，优良为：80-90，优秀为：>=90 要求输出课程号和选修⼈数，查询结果按⼈数降序排列，若⼈数相同，按课程号升序排列 ===
CId | Cname | 最高分 | 最低分 | 平均分 | 选修人数 | 及格率 | 中等率 | 优良率 | 优秀率
01 | 语文 | 80 | 31 | 64.5 | 6 | 0.6666666666666666 | 0.3333333333333333 | 0.3333333333333333 | 0.0
02 | 数学 | 90 | 30 | 72.66666666666667 | 6 | 0.8333333333333334 | 0.0 | 0.5 | 0.16666666666666666
03 | 英语 | 99 | 20 | 68.5 | 6 | 0.6666666666666666 | 0.0 | 0.3333333333333333 | 0.3333333333333333


18.按各科平均成绩进⾏排序，并显示排名， Score 重复时保留名次空缺

In [86]:
query_18 = """
SELECT 
    s2.CId, 
    s2.avg_sc, 
    COUNT(s1.avg_sc) AS rank 
FROM 
    (SELECT CId, ROUND(AVG(score), 2) AS avg_sc FROM SC GROUP BY CId) AS s1 
    JOIN 
    (SELECT CId, ROUND(AVG(score), 2) AS avg_sc FROM SC GROUP BY CId) AS s2 
    ON s1.avg_sc >= s2.avg_sc AND s1.CId = s1.CId 
GROUP BY 
    s2.CId, s2.avg_sc 
ORDER BY 
    rank;
"""
print_query_result("按各科平均成绩进⾏排序，并显示排名， Score 重复时保留名次空缺", query_18)


=== 按各科平均成绩进⾏排序，并显示排名， Score 重复时保留名次空缺 ===
CId | avg_sc | rank
02 | 72.67 | 1
03 | 68.5 | 2
01 | 64.5 | 3


19.按各科平均成绩进⾏排序，并显示排名， Score 重复时不保留名次空缺

In [87]:
query_19 = """
SELECT 
    b.CId, 
    b.avg_sc, 
    ROW_NUMBER() OVER (ORDER BY b.avg_sc DESC) AS rank 
FROM 
    (SELECT CId, ROUND(AVG(score), 2) AS avg_sc FROM SC GROUP BY CId) AS b;
"""
print_query_result("按各科平均成绩进⾏排序，并显示排名， Score 重复时不保留名次空缺", query_19)


=== 按各科平均成绩进⾏排序，并显示排名， Score 重复时不保留名次空缺 ===
CId | avg_sc | rank
02 | 72.67 | 1
03 | 68.5 | 2
01 | 64.5 | 3


20.查询学生的总成绩，并进⾏排名，总分重复时保留名次空缺

In [88]:

query_20 = """
SELECT 
    s2.sid, 
    s2.sum_sc, 
    COUNT(s1.sum_sc) AS rank 
FROM 
    (SELECT sid, SUM(score) AS sum_sc FROM SC GROUP BY sid) AS s1 
    JOIN 
    (SELECT sid, SUM(score) AS sum_sc FROM SC GROUP BY sid) AS s2 
    ON s1.sum_sc >= s2.sum_sc 
GROUP BY 
    s2.sid, s2.sum_sc 
ORDER BY 
    rank;
"""
print_query_result("查询学生的总成绩，并进⾏排名，总分重复时保留名次空缺", query_20)


=== 查询学生的总成绩，并进⾏排名，总分重复时保留名次空缺 ===
sid | sum_sc | rank
01 | 269 | 1
03 | 240 | 2
02 | 210 | 3
07 | 187 | 4
05 | 163 | 5
04 | 100 | 6
06 | 65 | 7


21.查询学生的总成绩，并进⾏排名，总分重复时不保留名次空缺

In [89]:
query_21 = """
SELECT 
    b.sid, 
    b.sum_sc, 
    ROW_NUMBER() OVER (ORDER BY b.sum_sc DESC) AS rank 
FROM 
    (SELECT sid, SUM(score) AS sum_sc FROM SC GROUP BY sid) AS b;
"""
print_query_result("查询学生的总成绩，并进⾏排名，总分重复时不保留名次空缺", query_21)


=== 查询学生的总成绩，并进⾏排名，总分重复时不保留名次空缺 ===
sid | sum_sc | rank
01 | 269 | 1
03 | 240 | 2
02 | 210 | 3
07 | 187 | 4
05 | 163 | 5
04 | 100 | 6
06 | 65 | 7


22.统计各科成绩各分数段人数：课程编号，课程名称，[100-85]，[85-70]，[70-60]，[60-0]及所占百分比

In [90]:
query_22 = """
SELECT 
    sc.CId, 
    c.Cname, 
    SUM(CASE WHEN sc.score > 85 AND sc.score <= 100 THEN 1 ELSE 0 END) AS '[100-85]', 
    SUM(CASE WHEN sc.score > 85 AND sc.score <= 100 THEN 1 ELSE 0 END) * 1.0 / COUNT(sc.CId) AS '百分比', 
    SUM(CASE WHEN sc.score > 70 AND sc.score <= 85 THEN 1 ELSE 0 END) AS '[85-70]', 
    SUM(CASE WHEN sc.score > 70 AND sc.score <= 85 THEN 1 ELSE 0 END) * 1.0 / COUNT(sc.CId) AS '百分比', 
    SUM(CASE WHEN sc.score > 60 AND sc.score <= 70 THEN 1 ELSE 0 END) AS '[70-60]', 
    SUM(CASE WHEN sc.score > 60 AND sc.score <= 70 THEN 1 ELSE 0 END) * 1.0 / COUNT(sc.CId) AS '百分比', 
    SUM(CASE WHEN sc.score > 0 AND sc.score <= 60 THEN 1 ELSE 0 END) AS '[60-0]', 
    SUM(CASE WHEN sc.score > 0 AND sc.score <= 60 THEN 1 ELSE 0 END) * 1.0 / COUNT(sc.CId) AS '百分比' 
FROM 
    SC 
    JOIN Course AS c ON sc.CId = c.CId 
GROUP BY 
    sc.CId, c.Cname;
    """
print_query_result("统计各科成绩各分数段人数：课程编号，课程名称，[100-85]，[85-70]，[70-60]，[60-0]及所占百分比", query_22)


=== 统计各科成绩各分数段人数：课程编号，课程名称，[100-85]，[85-70]，[70-60]，[60-0]及所占百分比 ===
CId | Cname | [100-85] | 百分比 | [85-70] | 百分比 | [70-60] | 百分比 | [60-0] | 百分比
01 | 语文 | 0 | 0.0 | 3 | 0.5 | 1 | 0.16666666666666666 | 2 | 0.3333333333333333
02 | 数学 | 3 | 0.5 | 1 | 0.16666666666666666 | 0 | 0.0 | 2 | 0.3333333333333333
03 | 英语 | 2 | 0.3333333333333333 | 2 | 0.3333333333333333 | 0 | 0.0 | 2 | 0.3333333333333333


23.查询各科成绩前三名的记录

In [91]:
query_23 = """
SELECT CId, score 
FROM (
    SELECT CId, score FROM SC WHERE CId = '01' 
    UNION ALL
    SELECT CId, score FROM SC WHERE CId = '02' 
    UNION ALL
    SELECT CId, score FROM SC WHERE CId = '03'
) 
ORDER BY CId, score DESC 
LIMIT 9;
"""
print_query_result("查询各科成绩前三名的记录", query_23)


=== 查询各科成绩前三名的记录 ===
CId | score
01 | 80
01 | 80
01 | 76
01 | 70
01 | 50
01 | 31
02 | 90
02 | 89
02 | 87


24.查询每门课程被选修的学生数

In [92]:
query_24 = """
SELECT CId, COUNT(CId) AS "选课人数" 
FROM SC 
GROUP BY CId;
"""
print_query_result("查询每门课程被选修的学生数", query_24)


=== 查询每门课程被选修的学生数 ===
CId | 选课人数
01 | 6
02 | 6
03 | 6


25.查询出只选修两门课程的学生学号和姓名

In [93]:
query_25 = """
SELECT s2.SId, s2.Sname 
FROM 
    (SELECT SId, COUNT(SId) AS "选修课程数" FROM SC GROUP BY SId) AS s1 
    JOIN Student AS s2 ON s1.SId = s2.SId 
WHERE 
    s1."选修课程数" = 2;
    """
print_query_result("查询出只选修两门课程的学生学号和姓名", query_25)


=== 查询出只选修两门课程的学生学号和姓名 ===
SId | Sname
05 | 周梅
06 | 吴兰
07 | 郑竹


26.查询男生、⼥生⼈数

In [94]:
query_26 = """
SELECT Ssex, COUNT(SId) 
FROM Student 
GROUP BY Ssex;
"""
print_query_result("查询男生、⼥生⼈数", query_26)


=== 查询男生、⼥生⼈数 ===
Ssex | COUNT(SId)
女 | 8
男 | 4


27.查询名字中含有「风」字的学生信息

In [95]:
query_27 = """
SELECT * 
FROM Student 
WHERE Sname LIKE '%风%';
"""
print_query_result("查询名字中含有「风」字的学生信息", query_27)


=== 查询名字中含有「风」字的学生信息 ===
SId | Sname | Sage | Ssex
03 | 孙风 | 1990-12-20 | 男


28.查询同名同姓学生名单，并统计同名⼈数

In [96]:
query_28 = """
SELECT Sname, COUNT(*) AS "人数" 
FROM Student 
GROUP BY Sname 
HAVING COUNT(*) >= 2;
"""
print_query_result("查询同名同姓学生名单，并统计同名⼈数", query_28)


=== 查询同名同姓学生名单，并统计同名⼈数 ===
Sname | 人数
李四 | 2


29.查询 1990 年出生的学生名单

In [97]:
query_29 = """
SELECT * 
FROM Student 
WHERE strftime('%Y', Sage) = '1990';
"""
print_query_result("查询同名同姓学生名单，并统计同名⼈数", query_29)


=== 查询同名同姓学生名单，并统计同名⼈数 ===
SId | Sname | Sage | Ssex
01 | 赵雷 | 1990-01-01 | 男
02 | 钱电 | 1990-12-21 | 男
03 | 孙风 | 1990-12-20 | 男
04 | 李云 | 1990-12-06 | 男


30.查询每门课程的平均成绩，结果按平均成绩降序排列，平均成绩相同时，按课程编号升序排列

In [98]:
query_30 = """
SELECT CId, AVG(score) 
FROM SC 
GROUP BY CId 
ORDER BY AVG(score), CId;
"""
print_query_result("查询每门课程的平均成绩，结果按平均成绩降序排列，平均成绩相同时，按课程编号升序排列", query_30)


=== 查询每门课程的平均成绩，结果按平均成绩降序排列，平均成绩相同时，按课程编号升序排列 ===
CId | AVG(score)
01 | 64.5
03 | 68.5
02 | 72.66666666666667


31.查询平均成绩⼤于等于 85 的所有学生的学号、姓名和平均成绩

In [99]:
query_31 = """
SELECT s1.SId, s1.Sname, s2."平均分" 
FROM Student AS s1 
JOIN 
    (SELECT SId, AVG(score) AS "平均分" FROM SC GROUP BY SId) AS s2 
ON s1.SId = s2.SId 
WHERE s2."平均分" >= 85;
"""
print_query_result("查询平均成绩⼤于等于 85 的所有学生的学号、姓名和平均成绩", query_31)


=== 查询平均成绩⼤于等于 85 的所有学生的学号、姓名和平均成绩 ===
SId | Sname | 平均分
01 | 赵雷 | 89.66666666666667
07 | 郑竹 | 93.5


32.查询课程名称为「数学」，且分数低于 60 的学生姓名和分数

In [100]:
query_32 = """
SELECT s2.Sname, s1.score 
FROM 
    Course AS c1 
    JOIN SC AS s1 ON c1.CId = s1.CId 
    JOIN Student AS s2 ON s1.SId = s2.SId 
WHERE 
    c1.Cname = '数学' AND s1.score < 60;
    """
print_query_result("查询课程名称为「数学」，且分数低于 60 的学生姓名和分数", query_32)


=== 查询课程名称为「数学」，且分数低于 60 的学生姓名和分数 ===
Sname | score
李云 | 30


33.查询所有学生的课程及分数情况（存在学生没成绩，没选课的情况）

In [101]:
query_33 = """
SELECT s2.Sname, s2.SId, c1.Cname, s1.score 
FROM 
    SC AS s1 
    RIGHT JOIN Student AS s2 ON s1.SId = s2.SId 
    LEFT JOIN Course AS c1 ON s1.CId = c1.CId 
GROUP BY 
    s2.SId, s2.Sname, c1.Cname, s1.score;
    """
print_query_result("查询所有学生的课程及分数情况", query_33)


=== 查询所有学生的课程及分数情况 ===
Sname | SId | Cname | score
赵雷 | 01 | 数学 | 90
赵雷 | 01 | 英语 | 99
赵雷 | 01 | 语文 | 80
钱电 | 02 | 数学 | 60
钱电 | 02 | 英语 | 80
钱电 | 02 | 语文 | 70
孙风 | 03 | 数学 | 80
孙风 | 03 | 英语 | 80
孙风 | 03 | 语文 | 80
李云 | 04 | 数学 | 30
李云 | 04 | 英语 | 20
李云 | 04 | 语文 | 50
周梅 | 05 | 数学 | 87
周梅 | 05 | 语文 | 76
吴兰 | 06 | 英语 | 34
吴兰 | 06 | 语文 | 31
郑竹 | 07 | 数学 | 89
郑竹 | 07 | 英语 | 98
张三 | 09 | None | None
李四 | 10 | None | None
李四 | 11 | None | None
赵六 | 12 | None | None
孙七 | 13 | None | None


34.查询任何⼀门课程成绩在 70 分以上的姓名、课程名称和分数

In [102]:
query_34 = """
SELECT s2.Sname, c1.Cname, s1.score 
FROM 
    SC AS s1 
    JOIN Student AS s2 ON s1.SId = s2.SId 
    JOIN Course AS c1 ON s1.CId = c1.CId 
WHERE 
    s1.score > 70;
    """
print_query_result("查询任何⼀门课程成绩在 70 分以上的姓名、课程名称和分数", query_34)


=== 查询任何⼀门课程成绩在 70 分以上的姓名、课程名称和分数 ===
Sname | Cname | score
赵雷 | 语文 | 80
赵雷 | 数学 | 90
赵雷 | 英语 | 99
钱电 | 英语 | 80
孙风 | 语文 | 80
孙风 | 数学 | 80
孙风 | 英语 | 80
周梅 | 语文 | 76
周梅 | 数学 | 87
郑竹 | 数学 | 89
郑竹 | 英语 | 98


35.查询不及格的课程

In [103]:
query_35 = """
SELECT s2.Sname, c1.Cname, s1.score 
FROM 
    SC AS s1 
    JOIN Student AS s2 ON s1.SId = s2.SId 
    JOIN Course AS c1 ON s1.CId = c1.CId 
WHERE 
    s1.score < 60;
    """
print_query_result("查询不及格的课程", query_35)


=== 查询不及格的课程 ===
Sname | Cname | score
李云 | 语文 | 50
李云 | 数学 | 30
李云 | 英语 | 20
吴兰 | 语文 | 31
吴兰 | 英语 | 34


36.查询课程编号为 01 且课程成绩在 80 分以上的学生的学号和姓名

In [104]:
query_36 = """
SELECT s1.SId, s2.Sname 
FROM 
    (SELECT SId, score FROM SC WHERE CId = '01') AS s1 
    JOIN Student AS s2 ON s1.SId = s2.SId 
WHERE 
    s1.score >= 80;
    """
print_query_result("查询课程编号为 01 且课程成绩在 80 分以上的学生的学号和姓名", query_36)


=== 查询课程编号为 01 且课程成绩在 80 分以上的学生的学号和姓名 ===
SId | Sname
01 | 赵雷
03 | 孙风


37.求每门课程的学生⼈数

In [105]:
query_37 = """
SELECT CId, COUNT(CId) AS "学生人数" 
FROM SC 
GROUP BY CId;
"""
print_query_result("求每门课程的学生⼈数", query_37)


=== 求每门课程的学生⼈数 ===
CId | 学生人数
01 | 6
02 | 6
03 | 6


38.成绩不重复，查询选修「张三」老师所授课程的学生中，成绩最⾼的学生信息及其成绩

In [106]:
query_38 = """
SELECT s2.Sname, s2.SId, s1.score 
FROM 
    Course AS c1 
    JOIN SC AS s1 ON c1.CId = s1.CId 
    JOIN Student AS s2 ON s1.SId = s2.SId 
    JOIN Teacher AS t1 ON c1.TId = t1.TId 
WHERE 
    t1.Tname = '张三' 
ORDER BY 
    s1.score DESC 
LIMIT 1;
"""
print_query_result("成绩不重复，查询选修「张三」老师所授课程的学生中，成绩最⾼的学生信息及其成绩", query_38)


=== 成绩不重复，查询选修「张三」老师所授课程的学生中，成绩最⾼的学生信息及其成绩 ===
Sname | SId | score
赵雷 | 01 | 90


39.成绩有重复的情况下，查询选修「张三」老师所授课程的学生中，成绩最⾼的学生信息及其成绩

In [107]:
query_39 = """
SELECT s2.Sname, s2.SId, s1.score 
FROM Student AS s2 
JOIN SC AS s1 ON s1.SId = s2.SId 
WHERE s1.score = (
    SELECT MAX(score) 
    FROM SC AS s1 
    WHERE CId = (
        SELECT c1.CId 
        FROM Course AS c1 
        JOIN Teacher AS t1 ON c1.TId = t1.TId 
        WHERE t1.Tname = '张三'
    )
);
"""
print_query_result("成绩有重复的情况下，查询选修「张三」老师所授课程的学生中，成绩最⾼的学生信息及其成绩", query_39)


=== 成绩有重复的情况下，查询选修「张三」老师所授课程的学生中，成绩最⾼的学生信息及其成绩 ===
Sname | SId | score
赵雷 | 01 | 90


40.查询不同课程成绩相同的学生的学生编号、课程编号、学生成绩

In [108]:
query_40 = """
SELECT SId, CId, score 
FROM SC 
WHERE SId = (
    SELECT SId 
    FROM (
        SELECT SId, score 
        FROM SC 
        GROUP BY SId, score
    ) AS s1 
    GROUP BY SId 
    HAVING COUNT(SId) = 1
);
"""
print_query_result("查询不同课程成绩相同的学生的学生编号、课程编号、学生成绩", query_40)


=== 查询不同课程成绩相同的学生的学生编号、课程编号、学生成绩 ===
SId | CId | score
03 | 01 | 80
03 | 02 | 80
03 | 03 | 80


41.查询每门课程成绩最好的前两名

In [109]:
query_41 = """
SELECT s1.* 
FROM SC s1 
WHERE 
    (
        SELECT COUNT(1) 
        FROM SC s2 
        WHERE 
            s1.CId = s2.CId AND s2.score >= s1.score
    ) <= 2 
ORDER BY 
    s1.CId, s1.score DESC;
    """
print_query_result("查询每门课程成绩最好的前两名", query_41)


=== 查询每门课程成绩最好的前两名 ===
SId | CId | score
01 | 01 | 80
03 | 01 | 80
01 | 02 | 90
07 | 02 | 89
01 | 03 | 99
07 | 03 | 98


42.统计每门课程的学生选修⼈数（超过 5 ⼈的课程才统计）

In [110]:
query_42 = """
SELECT CId, COUNT(CId) AS "学生人数" 
FROM SC 
GROUP BY CId 
HAVING COUNT(CId) > 5;
"""
print_query_result("统计每门课程的学生选修⼈数（超过 5 ⼈的课程才统计）", query_42)


=== 统计每门课程的学生选修⼈数（超过 5 ⼈的课程才统计） ===
CId | 学生人数
01 | 6
02 | 6
03 | 6


43.检索⾄少选修两门课程的学生学号

In [111]:
query_43 = """
SELECT CId, COUNT(CId) AS "学生人数" 
FROM SC 
GROUP BY CId 
HAVING COUNT(CId) >= 5;
"""
print_query_result("检索⾄少选修两门课程的学生学号", query_43)


=== 检索⾄少选修两门课程的学生学号 ===
CId | 学生人数
01 | 6
02 | 6
03 | 6


44.查询选修了全部课程的学生信息

In [112]:
query_44 = """
SELECT sid FROM sc GROUP BY SId HAVING count(sid)=3
"""
print_query_result("查询选修了全部课程的学生信息", query_44)


=== 查询选修了全部课程的学生信息 ===
SId
01
02
03
04


45.查询各学生的年龄，只按年份来算

In [113]:
query_45 = """
SELECT Sname, strftime('%Y', 'now') - strftime('%Y', Sage) AS "年龄" 
FROM Student;
"""
print_query_result("查询各学生的年龄，只按年份来算", query_45)


=== 查询各学生的年龄，只按年份来算 ===
Sname | 年龄
赵雷 | 35
钱电 | 35
孙风 | 35
李云 | 35
周梅 | 34
吴兰 | 33
郑竹 | 36
张三 | 8
李四 | 8
李四 | 13
赵六 | 12
孙七 | 11


46.按照出生日期来算，当前月日 < 出生年月的月日则，年龄减⼀

In [114]:
query_46 = """
SELECT 
    SId, Sname, Ssex, Sage, 
    strftime('%Y', 'now') - strftime('%Y', Sage) - (strftime('%m-%d', 'now') < strftime('%m-%d', Sage)) AS "按月日计算", 
    strftime('%Y', 'now') - strftime('%Y', Sage) AS "按年份计算" 
FROM Student;
"""
print_query_result("按照出生日期来算，当前月日 < 出生年月的月日则，年龄减⼀", query_46)


=== 按照出生日期来算，当前月日 < 出生年月的月日则，年龄减⼀ ===
SId | Sname | Ssex | Sage | 按月日计算 | 按年份计算
01 | 赵雷 | 男 | 1990-01-01 | 35 | 35
02 | 钱电 | 男 | 1990-12-21 | 34 | 35
03 | 孙风 | 男 | 1990-12-20 | 34 | 35
04 | 李云 | 男 | 1990-12-06 | 34 | 35
05 | 周梅 | 女 | 1991-12-01 | 33 | 34
06 | 吴兰 | 女 | 1992-01-01 | 33 | 33
07 | 郑竹 | 女 | 1989-01-01 | 36 | 36
09 | 张三 | 女 | 2017-12-20 | 7 | 8
10 | 李四 | 女 | 2017-12-25 | 7 | 8
11 | 李四 | 女 | 2012-06-06 | 12 | 13
12 | 赵六 | 女 | 2013-06-13 | 11 | 12
13 | 孙七 | 女 | 2014-06-01 | 10 | 11


47.查询本周过生日的学生

In [115]:
query_47 = """
SELECT SId, Sname, Ssex, Sage 
FROM Student 
WHERE strftime('%W', Sage) = strftime('%W', 'now');
"""
print_query_result("查询本周过生日的学生", query_47)


=== 查询本周过生日的学生 ===
SId | Sname | Ssex | Sage


48.查询下周过生日的学生

In [116]:
query_48 = """
SELECT SId, Sname, Ssex, Sage 
FROM Student 
WHERE strftime('%W', Sage) = strftime('%W', 'now', '+7 days');
"""
print_query_result("查询下周过生日的学生", query_48)


=== 查询下周过生日的学生 ===
SId | Sname | Ssex | Sage


49.查询本月过生日的学生

In [117]:
query_49 = """
SELECT * 
FROM Student 
WHERE strftime('%m', Sage) = strftime('%m', 'now');
"""
print_query_result("查询本月过生日的学生", query_49)


=== 查询本月过生日的学生 ===
SId | Sname | Sage | Ssex


50.查询下月过生日的学生

In [118]:
query_50 = """
SELECT * 
FROM Student 
WHERE strftime('%m', Sage) = strftime('%m', 'now', '+1 month');
"""
print_query_result("查询下月过生日的学生", query_50)


=== 查询下月过生日的学生 ===
SId | Sname | Sage | Ssex
